# ♟️ Chess Numba — Phase 8 Colab SPSA & Benchmark Runner
## 空步剪枝 (NMP) 體系 SF19 結構對齊、零調參前測與連續 SPSA

本 Notebook 執行 **Phase 8**：
1. **【Part 1: 零調參前測】**：以目前 HCE 預設常數，讓對齊 SF19 Step 10 新演算法的引擎直接對決舊引擎（`chess_engine.classical_old`），量測純演算法邏輯改善效果。
2. **【Part 2: 4 軸連續 SPSA】**：在 Colab（7 併發、`inprocess` 單次 JIT）高速搜尋 NMP 在 HCE 空間的最佳連續參數。
3. **【Part 3: 調參後終極決戰】**：套用 SPSA 最佳候選參數，與舊引擎進行長時控終極驗收（300,000 nodes）。

技術計畫書：`chess_engine/classical/PHASE8_NMP_SF19_ALIGNMENT_PLAN.md`

### 🎯 4 個連續 SPSA 調參軸（全為 `SearchContext.tune[]` 運行時槽位，免重編譯）
| 參數名稱 | 基準值 | 探索範圍 | 步長 c | tune 槽位 | 物理意義 |
| :-- | --: | :-: | --: | --: | :-- |
| `NMP_BASE` | 200 | [100, 350] | 12 | 26 | 靜態基礎門檻（控制 NMP 整體鬆緊尺度） |
| `NMP_DEPTH_COEF` | 14 | [0, 30] | 2 | 27 | 隨深度放寬門檻之斜率 |
| `NMP_IMP_COEF` | 45 | [0, 90] | 5 | 28 | 局勢改善時之放寬補貼（防守反擊保護） |
| `NMP_EVAL_MARGIN_DIV` | 150 | [64, 400] | 16 | 20 | 超額靜態估值換取動態減深 $R$ 之除數 |

> **注意**：舊引擎 `chess_engine.classical_old` 保持為對照基線（未包含 Phase 8 改動），請勿同步以免失去對照基準。


In [9]:
# 1. 檢視 Colab 硬體
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python3 --version

CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
Python 3.13.16


In [18]:
# 2. 設定（續跑時請勿更動 PARAMS / GAMES / NODES / CONCURRENCY / TOTAL_ITERS / SPSA_A_RATIO）
REPO_URL = "https://github.com/b08203031/chess_numba.git"
BRANCH = "feat-initial-bitboard-setup"
RUN_TAG = "nmp_p8_l1"

PARAMS = [
    "NMP_BASE",
    "NMP_DEPTH_COEF",
    "NMP_IMP_COEF",
    "NMP_EVAL_MARGIN_DIV",
]

# ---- Part 1: 零調參前測 (vs classical_old) ----
PRE_NODES = 200000
PRE_GAMES = 100          # 只作冒煙/無明顯退步檢查（95% CI 約 ±50 Elo），不作決策依據
PRE_TT_MB = 64

# ---- Part 2: L1 SPSA 雲端調參 ----
NODES = 50000           # 50k nodes：高速 SPSA 調參，每輪約 15-18 分鐘（Part 1 用 150k、Part 3 用 300k）
GAMES = 200              # 每輪 plus/minus 對局數
TOTAL_ITERS = 40        # 計畫輪數
CHUNK_ITERS = TOTAL_ITERS
TT_MB = 32
SPSA_A_RATIO = 0.2
CONCURRENCY = 7          # 8 vCPU -> 7 併發
TAIL = 0.3               # Polyak-Ruppert 尾端比例

# ---- Part 3: 調參後終極決戰 (vs classical_old) ----
L2_NODES = 300000
L2_GAMES = 600
L2_TT_MB = 64

USE_DRIVE = True         # 狀態/日誌/候選存到 Google Drive
RUN_SMOKE = False         # 第一次先跑 8 局冒煙（驗證指令與 JIT）

In [11]:
# 3. 持久化目錄（Google Drive）
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/chess_numba_p8"
else:
    PERSIST = "/content/chess_numba_p8_persist"

for sub in ("states", "logs", "results"):
    os.makedirs(f"{PERSIST}/{sub}", exist_ok=True)

STATE_FILE = f"{PERSIST}/states/{RUN_TAG}.json"
LOG_FILE = f"{PERSIST}/logs/{RUN_TAG}.jsonl"
CAND_UNIFORM = f"{PERSIST}/results/candidate_p8_tail_uniform.json"
CAND_LINEAR = f"{PERSIST}/results/candidate_p8_tail_linear.json"
print("persist dir:", PERSIST)
print("state:", STATE_FILE)
print("log  :", LOG_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
persist dir: /content/drive/MyDrive/chess_numba_p8
state: /content/drive/MyDrive/chess_numba_p8/states/nmp_p8_l1.json
log  : /content/drive/MyDrive/chess_numba_p8/logs/nmp_p8_l1.jsonl


In [12]:
# 4. 取得或更新程式碼倉庫
import os

# 防呆：若使用者未執行 Cell 2 即執行本格，自動補齊變數
if "BRANCH" not in globals():
    BRANCH = "feat-initial-bitboard-setup"
if "REPO_URL" not in globals():
    REPO_URL = "https://github.com/b08203031/chess_numba.git"

if not os.path.exists("/content/chess_numba"):
    !git clone -b {BRANCH} {REPO_URL} /content/chess_numba
else:
    !cd /content/chess_numba && git fetch origin {BRANCH} && git checkout -f {BRANCH} && git pull origin {BRANCH}

%cd /content/chess_numba
!git fetch origin {BRANCH}
!git checkout -f {BRANCH}
!git reset --hard origin/{BRANCH}
!pip install -q numba chess numpy matplotlib
!git log -1 --oneline


remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.76 KiB | 54.00 KiB/s, done.
From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
   571f5f6..bd16277  feat-initial-bitboard-setup -> origin/feat-initial-bitboard-setup
Already on 'feat-initial-bitboard-setup'
Your branch is behind 'origin/feat-initial-bitboard-setup' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
Updating 571f5f6..bd16277
Fast-forward
 chess_engine/classical/search.py |  23 ++--
 tune_colab_runner_p8.ipynb       | 277 ++++++++-------------------------------
 2 files changed, 67 insertions(+), 233 deletions(-)
/content/chess_numba
From ht

In [13]:
!cd /content/chess_numba && git fetch origin feat-initial-bitboard-setup && git reset --hard origin/feat-initial-bitboard-setup


From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
HEAD is now at bd16277 perf(search): restore lazy static evaluation in NMP to eliminate heavy full HCE bottleneck on non-passing cut nodes


In [14]:
# 5. 驗證 Phase 8 初始基線常數
import sys, json
sys.path.insert(0, "/content/chess_numba")

# 清除 Python 模組快取，確保 git pull 後一定載入磁碟上的最新檔案
for mod in list(sys.modules.keys()):
    if any(mod.startswith(pkg) for pkg in ("chess_engine", "tune_search", "tuner", "tools")):
        del sys.modules[mod]

from chess_engine.classical import constants as C
from tune_search.search_param_registry import get_default_params, get_spec

assert C.TUNE_SIZE == 42, f"TUNE_SIZE={C.TUNE_SIZE}; 需為 42"

expected_phase8_defaults = {
    "NMP_BASE": 200,
    "NMP_DEPTH_COEF": 14,
    "NMP_IMP_COEF": 45,
    "NMP_EVAL_MARGIN_DIV": 150,
}
baseline = get_default_params(PARAMS)
# Phase 8b 結構常數（SF19 對齊）— 確認 Colab 拉到的是新版程式碼
expected_struct = {
    "NMP_R_BASE": 7, "NMP_R_DIV": 3, "NMP_VERIFICATION_DEPTH": 8,
    "NMP_MIN_DEPTH": 1, "NMP_MIN_SIDE_NON_PAWNS": 2, "NMP_BETA_FLOOR": -960,
}
got_struct = {k: getattr(C, k) for k in expected_struct}
assert got_struct == expected_struct, f"結構常數不符（請 git pull）：{got_struct}"
assert baseline == expected_phase8_defaults, f"基線與 Phase 8 預期不一致！{baseline} vs {expected_phase8_defaults}"

BASELINE_JSON = "/content/chess_numba/tune_search/baseline_nmp_p8.json"
with open(BASELINE_JSON, "w", encoding="utf-8") as f:
    json.dump(baseline, f, indent=2)

print(f"{'param':30s} {'default':>8s} {'range':>14s} {'c':>5s} {'slot':>5s}")
for name in PARAMS:
    s = get_spec(name)
    print(f"{name:30s} {s.default:8d} {'['+str(s.minimum)+','+str(s.maximum)+']':>14s} {s.step:5d} {s.runtime_index:5d}")
print("✅ Phase 8 基線驗證通過；基線檔:", BASELINE_JSON)

param                           default          range     c  slot
NMP_BASE                            200      [120,350]    10    26
NMP_DEPTH_COEF                       14         [6,26]     2    27
NMP_IMP_COEF                         45        [15,75]     5    28
NMP_EVAL_MARGIN_DIV                 150      [120,450]    16    20
✅ Phase 8 基線驗證通過；基線檔: /content/chess_numba/tune_search/baseline_nmp_p8.json


## ⚔️ Part 1: 重構後零調參冒煙對戰 (Pre-Tuning Smoke vs `classical_old`)

**目的只有兩個：確認新引擎能正常跑完整局、且沒有明顯退步。** 100 局的 95% 信賴區間約 ±50 Elo，**不作為是否調參的決策依據**——無論結果如何都直接進入 Part 2。

新版 Step 10（Phase 8b，SF19 對齊）：
* 只在 `cutNode` 嘗試；門檻與 R 使用 SF19 的 `staticEval`（完整修正後靜態評估，不含 TT 修正）
* 門檻 `beta - 14*d - 45*imp + 200`；`R = 7 + d/3 + max(0, (staticEval-beta)/150)`（無上限）
* 移除非 SF 的額外條件（depth>=3、2 個非兵子力、|beta|<KNOWN_WIN、!is_win(static)）；beta 下限 -960（SF -2000 換算）
* `!is_win(nullValue)`；depth >= 16 才做驗證搜尋，`nmpMinPly` 局部封鎖（已修正 0 值旗標碰撞）

> 紅旗：若出現非法走步、崩潰、或勝率 < 35%，請停止並回報，不要進 Part 2。


In [30]:
# 6. Part 1 零調參冒煙對戰 ({PRE_GAMES} 局 @ {PRE_NODES} nodes vs classical_old)
!python tools/tournament.py \
  --engine1-module chess_engine.classical \
  --engine2-module chess_engine.classical_old \
  --name1 "New_SF19_NMP_PreTune" \
  --name2 "Old_Baseline" \
  --nodes {PRE_NODES} \
  --games {PRE_GAMES} \
  --concurrency {CONCURRENCY} \
  --tt-mb {PRE_TT_MB} \
  --backend inprocess \
  --pgn "{PERSIST}/results/pre_tune_match_vs_old.pgn"

Backend: inprocess; workers=7; TT=64 MB/context; openings=34700; seed=0
JIT plan: classical once + classical_old once; worker contexts share compiled code.
Warm-up: Kiwipete depth 14 for each Python/Numba backend, then one complete startpos game at depth 1 (all results discarded).
[pool] allocate worker 1/7
[warmup] New_SF19_NMP_PreTune: Kiwipete depth=14...
info depth 1 score cp 1 nodes 494 nps 1301 hashfull 0.0% time 379 pv e2a6
info depth 2 score cp 1 nodes 885 nps 1490 hashfull 0.0% time 593 pv e2a6 b4c3
info depth 3 score cp -34 nodes 1872 nps 3145 hashfull 0.0% time 595 pv e2a6 b4c3 d2c3
info depth 4 score cp -34 nodes 2312 nps 3881 hashfull 0.0% time 595 pv e2a6 b4c3 d2c3 h3g2
info depth 5 score cp -125 nodes 5843 nps 9742 hashfull 0.0% time 599 pv e2a6 b4c3 d2c3 e6d5 e4d5
info depth 6 score cp -125 nodes 6430 nps 10705 hashfull 0.0% time 600 pv e2a6 b4c3 d2c3 e6d5 e4d5 h3g2
info depth 7 score cp -91 nodes 9304 nps 15384 hashfull 0.0% time 604 pv e2a6 b4c3 d2c3 e6d5 e5g4 f6g4
in

## 🔬 Part 2: 雲端 SPSA 4 軸高速調參 (L1 SPSA 4-Axis Continuous Tuning)

在 Part 1 掌握演算法基線表現後，此處透過 SPSA 對 4 個連續門檻進行梯度探索，解開 HCE 尺度下的最優空步剪枝超參數。

In [36]:
# 7. SPSA 冒煙測試（8 局 @ 20k nodes；確認 inprocess worker 與 JIT 通路）
if RUN_SMOKE:
    !python -m tune_search.search_tuner \
      --params {",".join(PARAMS)} \
      --backend inprocess --scale-mode integer \
      --iter 1 --games 8 --nodes 20000 \
      --spsa-a-ratio {SPSA_A_RATIO} --concurrency {CONCURRENCY} \
      --adjudicate --log /tmp/p8_smoke.jsonl --state ''
else:
    print("RUN_SMOKE=False，略過。")

RUN_SMOKE=False，略過。


In [19]:
# 8. 執行 L1 SPSA（支援斷點續跑；一律傳 --schedule-iterations 以固定衰減時程）
import os, json

done_iters = 0
if os.path.exists(STATE_FILE):
    try:
        with open(STATE_FILE, "r", encoding="utf-8") as f:
            state_data = json.load(f)
        done_iters = int(state_data.get("iteration", 0))
        
        # 若使用者在續跑時動態調整了 NODES、GAMES 或 CONCURRENCY，自動同步 checkpoint 使其無縫續跑
        need_sync = False
        for k, v in [("nodes", NODES), ("games", GAMES), ("match_games", GAMES), ("concurrency", CONCURRENCY)]:
            if state_data.get(k) != v:
                print(f"[resume] 自動平滑升級 checkpoint {k}: {state_data.get(k)} -> {v}")
                state_data[k] = v
                need_sync = True
        if need_sync:
            with open(STATE_FILE, "w", encoding="utf-8") as f:
                json.dump(state_data, f, indent=2, ensure_ascii=False)
                
        resume_flag = "--resume"
    except Exception as exc:
        print(f"[resume] 讀取狀態檔異常: {exc}，將嘗試全新啟動")
        resume_flag = ""
else:
    resume_flag = ""

remaining = max(0, TOTAL_ITERS - done_iters)
run_now = min(remaining, CHUNK_ITERS)
print(f"已完成 {done_iters}/{TOTAL_ITERS} 輪；本次執行 {run_now} 輪（{'續跑' if resume_flag else '全新開跑'}）")
print(f"預估：{GAMES} 局 @ {NODES:,} nodes，約 {GAMES * 3.7 * NODES / 100000 / 60:.0f} 分鐘/輪（7 併發）")

if run_now > 0:
    !python -m tune_search.search_tuner \
      --params {",".join(PARAMS)} \
      --backend inprocess --scale-mode integer \
      --iter {run_now} --schedule-iterations {TOTAL_ITERS} \
      --games {GAMES} --nodes {NODES} --tt-mb {TT_MB} \
      --spsa-a-ratio {SPSA_A_RATIO} --concurrency {CONCURRENCY} \
      --adjudicate \
      --log {LOG_FILE} --state {STATE_FILE} {resume_flag}
else:
    print("✅ L1 全部輪數已完成，請繼續執行下一個 cell。")


[resume] 自動平滑升級 checkpoint nodes: 150000 -> 50000
[resume] 自動平滑升級 checkpoint games: 300 -> 200
[resume] 自動平滑升級 checkpoint match_games: 300 -> 200
已完成 7/40 輪；本次執行 33 輪（續跑）
預估：200 局 @ 50,000 nodes，約 6 分鐘/輪（7 併發）
[resume] iteration=7 from /content/drive/MyDrive/chess_numba_p8/states/nmp_p8_l1.json (inprocess)
[inprocess] openings=500 nodes=50000 concurrency=7 tt=32MB
[pool] allocate worker pair 1/7 (tt=32MB)...
[pool] allocate worker pair 2/7 (tt=32MB)...
[pool] allocate worker pair 3/7 (tt=32MB)...
[pool] allocate worker pair 4/7 (tt=32MB)...
[pool] allocate worker pair 5/7 (tt=32MB)...
[pool] allocate worker pair 6/7 (tt=32MB)...
[pool] allocate worker pair 7/7 (tt=32MB)...
[warmup] JIT nodes=5000 (may take several minutes first time)...
[warmup] done in 3.0s
[warmup] JIT nodes=1500 (may take several minutes first time)...
[warmup] done in 0.0s
[inprocess] touch worker 2/7
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0.0s
[warmup] JIT nodes=800 (may t

In [ ]:
# 9. 產出候選參數（Polyak-Ruppert 尾端平滑抽取）
!python tune_search/summarize_run.py {LOG_FILE} --tail {TAIL} --weight-mode uniform --out {CAND_UNIFORM}
!python tune_search/summarize_run.py {LOG_FILE} --tail {TAIL} --weight-mode linear  --out {CAND_LINEAR}
print("\n=== uniform candidate ===")
!cat {CAND_UNIFORM}
print("\n=== linear candidate ===")
!cat {CAND_LINEAR}

In [ ]:
# 10. 候選合理性審查與漂移監控
import json
if os.path.exists(CAND_UNIFORM):
    cand = json.load(open(CAND_UNIFORM, encoding="utf-8"))["averaged_params"]
    print(f"{'param':30s} {'baseline':>9s} {'candidate':>10s} {'delta%':>8s}  note")
    for name in PARAMS:
        s = get_spec(name)
        v = int(cand[name]); b = expected_phase8_defaults[name]
        notes = []
        if v <= s.minimum or v >= s.maximum:
            notes.append("⚠ 撞到範圍邊界")
        print(f"{name:30s} {b:9d} {v:10d} {100.0*(v-b)/b:+8.1f}  {' '.join(notes)}")

## 🏆 Part 3: 調參後終極決戰 vs 舊引擎 (Post-Tuning Final Battle vs `classical_old`)

在此步驟，我們將 SPSA 產出的最佳候選向量套入現行引擎，並在 **長時控（300,000 nodes, 64MB TT, 400 局）** 下與基準引擎 `chess_engine.classical_old` 進行決戰驗收！

In [ ]:
# 11. 將候選參數注入現行引擎環境，展開 Part 3 終極對戰
import json
cand_params = json.load(open(CAND_UNIFORM, encoding="utf-8"))["averaged_params"]
print("套用候選參數:", cand_params)

# 透過 --engine1-env 或專屬對抗模式與 classical_old 對抗
!python -m tune_search.search_param_match \
  --candidate {CAND_UNIFORM} \
  --baseline {BASELINE_JSON} \
  --nodes {L2_NODES} \
  --games {L2_GAMES} \
  --concurrency {CONCURRENCY} \
  --tt-mb {L2_TT_MB} \
  --sprt 0.0,3.0 \
  --backend inprocess

In [ ]:
# 12. （附加對照）候選版與未調參版直接 A/B 比較驗收
!python tools/tournament.py \
  --engine1-module chess_engine.classical \
  --engine2-module chess_engine.classical_old \
  --name1 "New_SF19_NMP_Tuned" \
  --name2 "Old_Baseline" \
  --nodes {L2_NODES} \
  --games {L2_GAMES} \
  --concurrency {CONCURRENCY} \
  --tt-mb {L2_TT_MB} \
  --backend inprocess \
  --pgn "{PERSIST}/results/final_post_tune_vs_old.pgn"

### 🏁 收官與落地步驟
1. 下載 Drive 中的 `candidate_p8_tail_uniform.json` 與對戰結果日誌。
2. 修改 `chess_engine/classical/constants.py` 的 `NMP_BASE` / `NMP_DEPTH_COEF` / `NMP_IMP_COEF` / `NMP_EVAL_MARGIN_DIV`。
3. 執行 `python -m tuner.sync_constants_snapshot`。
4. 更新 `SEARCH_ANALYSIS.md` 與 `PHASE8_NMP_SF19_ALIGNMENT_PLAN.md` 戰役章節。
5. 驗收完成後再使用 `python tools/sync_classical_old.py --sync` 同步舊基準。